# 01 Data Audit - MetroPT-3 Dataset

This notebook starts the implementation stage. It loads the MetroPT-3 dataset, checks basic structure, confirms timestamp coverage, checks missing values and duplicate timestamps, and saves evidence tables for the dissertation.

Before running this notebook, place the raw MetroPT-3 dataset file inside `data/raw/`.


## 1. Import libraries and define paths

In [14]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

print('Python executable:', sys.executable)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)

# Notebook is expected to run from the notebooks/ folder
BASE_DIR = Path('..').resolve()
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
OUTPUT_TABLES_DIR = BASE_DIR / 'outputs' / 'tables'
OUTPUT_FIGURES_DIR = BASE_DIR / 'outputs' / 'figures'

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Project folder:', BASE_DIR)
print('Raw data folder:', RAW_DATA_DIR)


Python executable: c:\Users\user\AppData\Local\Programs\Python\Python310\python.exe
pandas: 2.2.2
numpy: 1.26.4
Project folder: C:\Users\user\Desktop\metropt_predictive_maintenance_starter\metropt_predictive_maintenance_starter
Raw data folder: C:\Users\user\Desktop\metropt_predictive_maintenance_starter\metropt_predictive_maintenance_starter\data\raw


## 2. Find and load the dataset

The code below searches `data/raw/` for a CSV, TXT or Excel file. If there are multiple files, it uses the first one. Rename your raw file clearly if needed.


In [15]:
candidate_files = []
for pattern in ['*.csv', '*.txt', '*.xlsx', '*.xls']:
    candidate_files.extend(RAW_DATA_DIR.glob(pattern))

if not candidate_files:
    raise FileNotFoundError(
        f'No dataset file found in {RAW_DATA_DIR}. Download MetroPT-3 and place the raw file inside data/raw/.'
    )

data_path = candidate_files[0]
print('Loading:', data_path.name)

if data_path.suffix.lower() in ['.xlsx', '.xls']:
    df = pd.read_excel(data_path)
else:
    df = pd.read_csv(data_path)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
df.head()


Loading: MetroPT3(AirCompressor).csv
Dataset loaded successfully.
Shape: (1516948, 17)


,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


## 3. Inspect columns and data types

In [16]:
columns_df = pd.DataFrame({
    'column_name': df.columns,
    'data_type': [str(dtype) for dtype in df.dtypes]
})
columns_df.to_csv(OUTPUT_TABLES_DIR / 'column_data_types.csv', index=False)
columns_df


,column_name,data_type
0,Unnamed: 0,int64
1,timestamp,object
2,TP2,float64
3,TP3,float64
4,H1,float64
5,DV_pressure,float64
6,Reservoirs,float64
7,Oil_temperature,float64
8,Motor_current,float64
9,COMP,float64


## 4. Detect timestamp column and check date range

In [17]:
possible_timestamp_cols = [
    col for col in df.columns
    if col.lower() in ['timestamp', 'time', 'datetime', 'date'] or 'time' in col.lower() or 'date' in col.lower()
]

print('Possible timestamp columns:', possible_timestamp_cols)

# Change this manually if the detected column is wrong.
timestamp_col = possible_timestamp_cols[0] if possible_timestamp_cols else 'timestamp'
print('Using timestamp column:', timestamp_col)

if timestamp_col not in df.columns:
    raise KeyError('Timestamp column not found. Please set timestamp_col manually to the correct column name.')

df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')
df = df.sort_values(timestamp_col).reset_index(drop=True)

print('Start date:', df[timestamp_col].min())
print('End date:', df[timestamp_col].max())
print('Invalid timestamps:', df[timestamp_col].isna().sum())
print('Duplicate timestamps:', df.duplicated(subset=timestamp_col).sum())


Possible timestamp columns: ['timestamp']
Using timestamp column: timestamp
Start date: 2020-02-01 00:00:00
End date: 2020-09-01 03:59:50
Invalid timestamps: 0
Duplicate timestamps: 0


## 5. Save dataset summary

In [18]:
summary_df = pd.DataFrame({
    'rows': [df.shape[0]],
    'columns': [df.shape[1]],
    'start_date': [df[timestamp_col].min()],
    'end_date': [df[timestamp_col].max()],
    'invalid_timestamps': [df[timestamp_col].isna().sum()],
    'duplicate_timestamps': [df.duplicated(subset=timestamp_col).sum()]
})

summary_df.to_csv(OUTPUT_TABLES_DIR / 'dataset_summary.csv', index=False)
summary_df


,rows,columns,start_date,end_date,invalid_timestamps,duplicate_timestamps
0,1516948,17,2020-02-01,2020-09-01 03:59:50,0,0


## 6. Missing value audit

In [19]:
missing_values = df.isnull().sum().reset_index()
missing_values.columns = ['column', 'missing_count']
missing_values['missing_percentage'] = (missing_values['missing_count'] / len(df)) * 100
missing_values = missing_values.sort_values('missing_count', ascending=False)

missing_values.to_csv(OUTPUT_TABLES_DIR / 'missing_values_summary.csv', index=False)
missing_values


,column,missing_count,missing_percentage
0,Unnamed: 0,0,0.0
9,COMP,0,0.0
15,Oil_level,0,0.0
14,Pressure_switch,0,0.0
13,LPS,0,0.0
12,MPG,0,0.0
11,Towers,0,0.0
10,DV_eletric,0,0.0
8,Motor_current,0,0.0
1,timestamp,0,0.0


## 7. Documented failure events

These documented failure intervals are used later for early-warning labels and event-level validation.


In [20]:
failure_events = pd.DataFrame({
    'event_id': ['F1', 'F2', 'F3', 'F4'],
    'failure_type': ['Air leak', 'Air leak', 'Air leak', 'Air leak'],
    'failure_start': [
        '2020-04-18 00:00',
        '2020-05-29 23:30',
        '2020-06-05 10:00',
        '2020-07-15 14:30'
    ],
    'failure_end': [
        '2020-04-18 23:59',
        '2020-05-30 06:00',
        '2020-06-07 14:30',
        '2020-07-15 19:00'
    ]
})

failure_events['failure_start'] = pd.to_datetime(failure_events['failure_start'])
failure_events['failure_end'] = pd.to_datetime(failure_events['failure_end'])

failure_events['inside_dataset_range'] = (
    (failure_events['failure_start'] >= df[timestamp_col].min()) &
    (failure_events['failure_end'] <= df[timestamp_col].max())
)

failure_events.to_csv(OUTPUT_TABLES_DIR / 'documented_failure_events.csv', index=False)
failure_events


,event_id,failure_type,failure_start,failure_end,inside_dataset_range
0,F1,Air leak,2020-04-18 00:00:00,2020-04-18 23:59:00,True
1,F2,Air leak,2020-05-29 23:30:00,2020-05-30 06:00:00,True
2,F3,Air leak,2020-06-05 10:00:00,2020-06-07 14:30:00,True
3,F4,Air leak,2020-07-15 14:30:00,2020-07-15 19:00:00,True


## 8. Separate numerical and non-numerical variables

In [21]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in df.columns if c not in numeric_cols]

variable_groups = pd.DataFrame({
    'group': ['numeric'] * len(numeric_cols) + ['non_numeric_or_datetime'] * len(non_numeric_cols),
    'column': numeric_cols + non_numeric_cols
})

variable_groups.to_csv(OUTPUT_TABLES_DIR / 'variable_groups.csv', index=False)
variable_groups


,group,column
0,numeric,Unnamed: 0
1,numeric,TP2
2,numeric,TP3
3,numeric,H1
4,numeric,DV_pressure
5,numeric,Reservoirs
6,numeric,Oil_temperature
7,numeric,Motor_current
8,numeric,COMP
9,numeric,DV_eletric


## 9. Audit outputs created

After running this notebook, check `outputs/tables/` for:

- `dataset_summary.csv`
- `missing_values_summary.csv`
- `column_data_types.csv`
- `documented_failure_events.csv`
- `variable_groups.csv`
